# Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from collections import Counter
import warnings
import os

warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_fig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  [Saved] {path}")

print("=" * 70)
print("  ROBUST RECOMMENDER SYSTEM – INTERIM ANALYSIS")
print("=" * 70)

  ROBUST RECOMMENDER SYSTEM – INTERIM ANALYSIS


# Data Loading

In [4]:
print("\n[1] LOADING DATA")
print("-" * 40)

movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
users   = pd.read_csv("users.csv")

# Convert timestamp to datetime
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

print(f"  Movies  : {movies.shape[0]:,} rows × {movies.shape[1]} cols")
print(f"  Ratings : {ratings.shape[0]:,} rows × {ratings.shape[1]} cols")
print(f"  Users   : {users.shape[0]:,} rows × {users.shape[1]} cols")


[1] LOADING DATA
----------------------------------------
  Movies  : 3,883 rows × 3 cols
  Ratings : 1,000,209 rows × 4 cols
  Users   : 6,040 rows × 5 cols


# Data Preprocessing

In [5]:
print("\n[2] DATA PREPROCESSING")
print("-" * 40)

# Missing value check 
print("\n  2.1 Missing Value Check")
for name, df in [("movies", movies), ("ratings", ratings), ("users", users)]:
    missing = df.isnull().sum()
    total_missing = missing.sum()
    print(f"    {name:10s}: {total_missing} missing values"
          + (f" → {missing[missing>0].to_dict()}" if total_missing else " ✓"))

# Duplicate detection
print("\n  2.2 Duplicate Detection")
dup_ratings = ratings.duplicated(subset=['UserID','MovieID']).sum()
print(f"    Duplicate (UserID, MovieID) pairs in ratings: {dup_ratings:,}")
if dup_ratings > 0:
    ratings = ratings.drop_duplicates(subset=['UserID','MovieID'], keep='last')
    print(f"    → Duplicates removed. Remaining ratings: {len(ratings):,}")

# Data type casting
print("\n  2.3 Data Type Casting")
ratings['Rating'] = ratings['Rating'].astype(float)
users['Age']      = users['Age'].astype(int)
print("    Rating cast to float, Age to int ✓")

# Genre expansion 
print("\n  2.4 Genre Expansion")
movies['GenreList'] = movies['Genres'].str.split('|')
all_genres = sorted(set(g for gl in movies['GenreList'] for g in gl))
print(f"    Unique genres: {len(all_genres)}")
print(f"    Genre list: {all_genres}")

# Age group mapping 
age_map = {1: "Under 18", 18: "18-24", 25: "25-34", 35: "35-44",
           45: "45-49", 50: "50-55", 56: "56+"}
users['AgeGroup'] = users['Age'].map(age_map)

# Occupation mapping 
occ_map = {
    0: "other/not specified", 1: "academic/educator", 2: "artist",
    3: "clerical/admin", 4: "college/grad student", 5: "customer service",
    6: "doctor/health care", 7: "executive/managerial", 8: "farmer",
    9: "homemaker", 10: "K-12 student", 11: "lawyer", 12: "programmer",
    13: "retired", 14: "sales/marketing", 15: "scientist", 16: "self-employed",
    17: "technician/engineer", 18: "tradesman/craftsman", 19: "unemployed",
    20: "writer"
}
users['OccupationName'] = users['Occupation'].map(occ_map)
print(f"    Age groups and occupation labels mapped ✓")

# Merge into master dataframe
print("\n  2.7 Building Master DataFrame")
master = (ratings
          .merge(users[['UserID','Gender','AgeGroup','OccupationName']], on='UserID')
          .merge(movies[['MovieID','Title','Genres','GenreList']], on='MovieID'))
print(f"    Master shape: {master.shape[0]:,} rows × {master.shape[1]} cols")

# Summary statistics 
print("\n  2.8 Summary Statistics – Ratings")
print(ratings['Rating'].describe().to_string())

print("\n  Data preprocessing complete ✓")


[2] DATA PREPROCESSING
----------------------------------------

  2.1 Missing Value Check
    movies    : 0 missing values ✓
    ratings   : 0 missing values ✓
    users     : 0 missing values ✓

  2.2 Duplicate Detection
    Duplicate (UserID, MovieID) pairs in ratings: 0

  2.3 Data Type Casting
    Rating cast to float, Age to int ✓

  2.4 Genre Expansion
    Unique genres: 18
    Genre list: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    Age groups and occupation labels mapped ✓

  2.7 Building Master DataFrame
    Master shape: 1,000,209 rows × 10 cols

  2.8 Summary Statistics – Ratings
count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00

  Data preprocessing complete ✓


# Save Merged Master Dataset to CSV

In [6]:
# ── Save Master DataFrame to CSV ─────────────────────────────────────────────
# After merging all three datasets (ratings + users + movies) into one master
# dataframe, we save it as a CSV file in the current working directory.
# This file serves as the clean ground truth before any noise injection.

print("\n[3] SAVING MERGED MASTER DATASET")
print("-" * 40)

# Drop GenreList (list column — not CSV-friendly) for the saved file
master_csv = master.drop(columns=['GenreList'])

# Save to current working directory
MASTER_CSV_PATH = "master_merged.csv"
master_csv.to_csv(MASTER_CSV_PATH, index=False)

print(f"  master_merged.csv saved ✓")
print(f"  Shape  : {master_csv.shape[0]:,} rows × {master_csv.shape[1]} columns")
print(f"  Columns: {master_csv.columns.tolist()}")
print(f"  Path   : ./{MASTER_CSV_PATH}")
print(f"\n  Preview:")
print(master_csv.head(3).to_string())


[3] SAVING MERGED MASTER DATASET
----------------------------------------
  master_merged.csv saved ✓
  Shape  : 1,000,209 rows × 9 columns
  Columns: ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'AgeGroup', 'OccupationName', 'Title', 'Genres']
  Path   : ./master_merged.csv

  Preview:
   UserID  MovieID  Rating           Timestamp Gender  AgeGroup OccupationName                                   Title                        Genres
0       1     1193     5.0 2000-12-31 22:12:40      F  Under 18   K-12 student  One Flew Over the Cuckoo's Nest (1975)                         Drama
1       1      661     3.0 2000-12-31 22:35:09      F  Under 18   K-12 student        James and the Giant Peach (1996)  Animation|Children's|Musical
2       1      914     3.0 2000-12-31 22:32:48      F  Under 18   K-12 student                     My Fair Lady (1964)               Musical|Romance


# Descriptive Analysis and Visualisation

## Rating Distribution

In [7]:
print("\n  3.1 Rating Distribution")
rating_counts = ratings['Rating'].value_counts().sort_index()
print(rating_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=sns.color_palette("Blues_d", 5), edgecolor='white', width=0.6)
ax.bar_label(bars, fmt=lambda x: f"{x/1e6:.2f}M", padding=3, fontsize=9)
ax.set_xlabel("Rating Value")
ax.set_ylabel("Number of Ratings")
ax.set_title("Figure 3.1 – Rating Distribution (MovieLens 1M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
save_fig("fig3_1_rating_distribution.png")


  3.1 Rating Distribution
Rating
1.0     56174
2.0    107557
3.0    261197
4.0    348971
5.0    226310
  [Saved] figures\fig3_1_rating_distribution.png


## Rating Per User (Activity Distribution)

In [8]:
print("\n  3.2 User Activity Distribution")
user_activity = ratings.groupby('UserID')['Rating'].count()
print(f"    Mean ratings/user : {user_activity.mean():.1f}")
print(f"    Median            : {user_activity.median():.0f}")
print(f"    Std               : {user_activity.std():.1f}")
print(f"    Min / Max         : {user_activity.min()} / {user_activity.max():,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_activity, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title("Figure 3.2a – Ratings per User (Full)")
axes[0].set_xlabel("Ratings per User"); axes[0].set_ylabel("Count")

axes[1].hist(user_activity[user_activity < 200], bins=60, color='steelblue', edgecolor='white')
axes[1].set_title("Figure 3.2b – Ratings per User (< 200, zoomed)")
axes[1].set_xlabel("Ratings per User"); axes[1].set_ylabel("Count")
plt.tight_layout()
save_fig("fig3_2_user_activity.png")


  3.2 User Activity Distribution
    Mean ratings/user : 165.6
    Median            : 96
    Std               : 192.7
    Min / Max         : 20 / 2,314
  [Saved] figures\fig3_2_user_activity.png


## Ratings per movie (item popularity)

In [9]:
print("\n  3.3 Item Popularity Distribution")
item_pop = ratings.groupby('MovieID')['Rating'].count()
print(f"    Mean ratings/movie : {item_pop.mean():.1f}")
print(f"    Median             : {item_pop.median():.0f}")
print(f"    Std                : {item_pop.std():.1f}")
print(f"    Min / Max          : {item_pop.min()} / {item_pop.max():,}")

# Long-tail analysis
top_10pct_movies = item_pop.nlargest(int(len(item_pop) * 0.1))
top_10pct_ratings = top_10pct_movies.sum()
print(f"\n    Top 10% movies account for "
      f"{top_10pct_ratings/len(ratings)*100:.1f}% of all ratings  (long-tail effect)")

fig, ax = plt.subplots(figsize=(8, 4))
sorted_pop = item_pop.sort_values(ascending=False).reset_index(drop=True)
ax.plot(sorted_pop.values, color='tomato', linewidth=1.5)
ax.fill_between(range(len(sorted_pop)), sorted_pop.values, alpha=0.15, color='tomato')
ax.set_title("Figure 3.3 – Item Popularity (Long-Tail Distribution)")
ax.set_xlabel("Movie Rank (by popularity)"); ax.set_ylabel("Number of Ratings")
ax.set_yscale('log')
save_fig("fig3_3_item_popularity.png")


  3.3 Item Popularity Distribution
    Mean ratings/movie : 269.9
    Median             : 124
    Std                : 384.0
    Min / Max          : 1 / 3,428

    Top 10% movies account for 44.4% of all ratings  (long-tail effect)
  [Saved] figures\fig3_3_item_popularity.png


## Genre Analysis

In [10]:
print("\n  3.4 Genre Analysis")
genre_counts = Counter(g for gl in movies['GenreList'] for g in gl)
genre_df = pd.DataFrame(genre_counts.items(), columns=['Genre','Count']).sort_values('Count', ascending=False)
print(genre_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=genre_df, x='Count', y='Genre', palette='Blues_r', ax=ax)
ax.set_title("Figure 3.4 – Movie Genre Distribution")
ax.set_xlabel("Number of Movies"); ax.set_ylabel("Genre")
plt.tight_layout()
save_fig("fig3_4_genre_distribution.png")


  3.4 Genre Analysis
      Genre  Count
      Drama   1603
     Comedy   1200
     Action    503
   Thriller    492
    Romance    471
     Horror    343
  Adventure    283
     Sci-Fi    276
 Children's    251
      Crime    211
        War    143
Documentary    127
    Musical    114
    Mystery    106
  Animation    105
    Fantasy     68
    Western     68
  Film-Noir     44
  [Saved] figures\fig3_4_genre_distribution.png


## Average Rating by Genre

In [11]:
print("\n  3.5 Average Rating by Genre")
master_exploded = master.copy()
master_exploded = master_exploded.explode('GenreList')
genre_rating = (master_exploded.groupby('GenreList')['Rating']
                .agg(['mean','count'])
                .rename(columns={'mean':'AvgRating','count':'NumRatings'})
                .sort_values('AvgRating', ascending=False))
print(genre_rating.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=genre_rating['AvgRating'], y=genre_rating.index, palette='RdYlGn', ax=ax)
ax.set_xlim(3.0, 4.2)
ax.axvline(x=genre_rating['AvgRating'].mean(), color='black', linestyle='--', linewidth=1, label='Mean')
ax.legend()
ax.set_title("Figure 3.5 – Average Rating by Genre")
ax.set_xlabel("Average Rating"); ax.set_ylabel("Genre")
plt.tight_layout()
save_fig("fig3_5_avg_rating_by_genre.png")


  3.5 Average Rating by Genre
             AvgRating  NumRatings
GenreList                         
Film-Noir        4.075       18261
Documentary      3.933        7910
War              3.893       68527
Drama            3.766      354529
Crime            3.709       79541
Animation        3.685       43293
Mystery          3.668       40178
Musical          3.666       41533
Western          3.638       20683
Romance          3.607      147523
Thriller         3.570      189680
Comedy           3.522      356580
Action           3.491      257457
Adventure        3.477      133953
Sci-Fi           3.467      157294
Fantasy          3.447       36301
Children's       3.422       72186
Horror           3.215       76386
  [Saved] figures\fig3_5_avg_rating_by_genre.png


## User Demographics

In [12]:
# ── 3.6 User Demographics ─────────────────────────────────────────────────────
print("\n  3.6 User Demographics")

gender_counts = users['Gender'].value_counts()
age_order     = ["Under 18","18-24","25-34","35-44","45-49","50-55","56+"]
age_counts    = users['AgeGroup'].value_counts().reindex(age_order, fill_value=0)
top_occ       = users['OccupationName'].value_counts().head(10)

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(16, 5))

gender_labels = ['Male' if g == 'M' else 'Female' for g in gender_counts.index]
ax0.pie(gender_counts.values, labels=gender_labels, autopct='%1.1f%%',
        colors=['#4C72B0','#DD8452'], startangle=90)
ax0.set_title("Figure 3.6a – Gender Distribution")

sns.barplot(x=age_counts.values, y=age_counts.index, palette='viridis', ax=ax1)
ax1.set_title("Figure 3.6b – Age Group Distribution")
ax1.set_xlabel("Number of Users"); ax1.set_ylabel("Age Group")

sns.barplot(x=top_occ.values, y=top_occ.index, palette='rocket_r', ax=ax2)
ax2.set_title("Figure 3.6c – Top 10 Occupations")
ax2.set_xlabel("Number of Users"); ax2.set_ylabel("Occupation")

plt.tight_layout()
save_fig("fig3_6_user_demographics.png")


  3.6 User Demographics
  [Saved] figures\fig3_6_user_demographics.png


## Rating Over time

In [13]:
print("\n  3.7 Rating Activity Over Time")
ratings['YearMonth'] = ratings['Timestamp'].dt.to_period('M')
time_activity = ratings.groupby('YearMonth').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_activity.index.astype(str), time_activity.values, color='#2E86AB', linewidth=1.5)
ax.fill_between(range(len(time_activity)), time_activity.values, alpha=0.15, color='#2E86AB')
ax.set_title("Figure 3.7 – Ratings Activity Over Time")
ax.set_xlabel("Year-Month"); ax.set_ylabel("Number of Ratings")
n = max(1, len(time_activity) // 8)
ax.set_xticks(range(0, len(time_activity), n))
ax.set_xticklabels([str(t) for t in time_activity.index[::n]], rotation=45, ha='right')
plt.tight_layout()
save_fig("fig3_7_ratings_over_time.png")


  3.7 Rating Activity Over Time
  [Saved] figures\fig3_7_ratings_over_time.png


# Natural Noise Indicators

## Rating Variance Per User

In [14]:
print("\n[4] NATURAL NOISE INDICATORS")
print("-" * 40)

print("\n  4.1 Rating Variance per User (Inconsistency Indicator)")
user_stats = ratings.groupby('UserID')['Rating'].agg(['mean','std','count'])
user_stats.columns = ['AvgRating','StdRating','NumRatings']
user_stats = user_stats[user_stats['NumRatings'] >= 5]  # min 5 ratings

high_variance_users = (user_stats['StdRating'] > user_stats['StdRating'].quantile(0.90))
print(f"    Total users with ≥5 ratings : {len(user_stats):,}")
print(f"    Mean rating std per user    : {user_stats['StdRating'].mean():.3f}")
print(f"    High-variance users (top 10%): {high_variance_users.sum():,}")
print(f"    → These are natural noise candidates (inconsistent rating behaviour)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_stats['StdRating'].dropna(), bins=50, color='#E76F51', edgecolor='white')
axes[0].axvline(user_stats['StdRating'].quantile(0.90), color='black', linestyle='--', label='90th pct')
axes[0].legend()
axes[0].set_title("Figure 4.1a – Rating Std Dev per User")
axes[0].set_xlabel("Std Dev of Ratings"); axes[0].set_ylabel("Number of Users")

axes[1].scatter(user_stats['NumRatings'], user_stats['StdRating'],
                alpha=0.2, s=5, color='#457B9D')
axes[1].set_title("Figure 4.1b – Rating Activity vs Variance")
axes[1].set_xlabel("Number of Ratings"); axes[1].set_ylabel("Rating Std Dev")
axes[1].set_xscale('log')
plt.tight_layout()
save_fig("fig4_1_user_variance.png")


[4] NATURAL NOISE INDICATORS
----------------------------------------

  4.1 Rating Variance per User (Inconsistency Indicator)
    Total users with ≥5 ratings : 6,040
    Mean rating std per user    : 1.010
    High-variance users (top 10%): 604
    → These are natural noise candidates (inconsistent rating behaviour)
  [Saved] figures\fig4_1_user_variance.png


## Extreme Raters - all 1 or all 5

In [15]:
print("\n  4.2 Extreme Raters (Polar Bimodal Pattern)")
all_high = (user_stats['AvgRating'] >= 4.5) & (user_stats['StdRating'] < 0.5)
all_low  = (user_stats['AvgRating'] <= 1.5) & (user_stats['StdRating'] < 0.5)
print(f"    Always-high raters (avg≥4.5, std<0.5) : {all_high.sum():,}")
print(f"    Always-low  raters (avg≤1.5, std<0.5) : {all_low.sum():,}")
print(f"    → Extreme raters introduce natural noise (lack of discrimination)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_stats['AvgRating'], bins=40, color='#457B9D', edgecolor='white')
ax.axvline(1.5, color='red',   linestyle='--', linewidth=1.5, label='Low threshold (1.5)')
ax.axvline(4.5, color='green', linestyle='--', linewidth=1.5, label='High threshold (4.5)')
ax.set_title("Figure 4.2 – Distribution of Per-User Average Ratings")
ax.set_xlabel("Average Rating per User"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_2_extreme_raters.png")


  4.2 Extreme Raters (Polar Bimodal Pattern)
    Always-high raters (avg≥4.5, std<0.5) : 11
    Always-low  raters (avg≤1.5, std<0.5) : 2
    → Extreme raters introduce natural noise (lack of discrimination)
  [Saved] figures\fig4_2_extreme_raters.png


## Rating Entropy Per User

In [16]:
print("\n  4.3 Rating Entropy per User")
def rating_entropy(ratings_series):
    counts = ratings_series.value_counts(normalize=True)
    return -np.sum(counts * np.log2(counts + 1e-10))

user_entropy = ratings.groupby('UserID')['Rating'].apply(rating_entropy)
print(f"    Mean entropy: {user_entropy.mean():.3f} (max possible = log2(5) ≈ 2.32)")
print(f"    Low entropy (<0.5) users: {(user_entropy < 0.5).sum():,}  → narrow rating range (noise risk)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_entropy, bins=40, color='#2A9D8F', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', label='Low-entropy threshold')
ax.set_title("Figure 4.3 – Rating Entropy per User")
ax.set_xlabel("Shannon Entropy of Rating Distribution"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_3_rating_entropy.png")


  4.3 Rating Entropy per User
    Mean entropy: 1.853 (max possible = log2(5) ≈ 2.32)
    Low entropy (<0.5) users: 7  → narrow rating range (noise risk)
  [Saved] figures\fig4_3_rating_entropy.png


## Temporal Inconsistancy: Rating Drift

In [17]:
print("\n  4.4 Temporal Rating Drift (Natural Noise)")
# Find users with ≥20 ratings; compute correlation of rating with time
user_drift = []
user_ids_with_enough = ratings.groupby('UserID').filter(lambda x: len(x) >= 20)['UserID'].unique()

np.random.seed(42)
sample_users = np.random.choice(user_ids_with_enough, size=min(500, len(user_ids_with_enough)), replace=False)

for uid in sample_users:
    u_ratings = ratings[ratings['UserID'] == uid].sort_values('Timestamp')
    ts = u_ratings['Timestamp'].astype(np.int64)
    r  = u_ratings['Rating'].values
    if len(r) >= 3 and ts.std() > 0:
        corr, _ = stats.spearmanr(ts, r)
        user_drift.append(corr)

user_drift = np.array(user_drift)
strong_drift = np.abs(user_drift) > 0.3
print(f"    Users sampled for drift analysis : {len(user_drift):,}")
print(f"    Users with |drift correlation| > 0.3 : {strong_drift.sum():,} ({strong_drift.mean()*100:.1f}%)")
print(f"    → Rating drift over time is a key natural noise indicator")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_drift, bins=40, color='#F4A261', edgecolor='white')
ax.axvline(-0.3, color='red', linestyle='--', label='|Drift| > 0.3')
ax.axvline( 0.3, color='red', linestyle='--')
ax.set_title("Figure 4.4 – Temporal Rating Drift (Spearman Corr)")
ax.set_xlabel("Spearman Correlation (Timestamp vs Rating)"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig4_4_temporal_drift.png")


  4.4 Temporal Rating Drift (Natural Noise)
    Users sampled for drift analysis : 500
    Users with |drift correlation| > 0.3 : 88 (17.6%)
    → Rating drift over time is a key natural noise indicator
  [Saved] figures\fig4_4_temporal_drift.png


# Malicious Noise Indicators

## Rating Burst Detection

In [18]:
print("\n  5.1 Rating Burst Detection (Temporal Clustering)")
# Flag users who rate many movies within a short time window (e.g., 1 hour)
ratings_sorted = ratings.sort_values(['UserID', 'Timestamp'])
ratings_sorted['TimeDiff'] = (ratings_sorted.groupby('UserID')['Timestamp']
                               .diff().dt.total_seconds().fillna(9999))

burst_threshold_sec = 3600  # 1 hour
burst_stats = (ratings_sorted[ratings_sorted['TimeDiff'] < burst_threshold_sec]
               .groupby('UserID').size()
               .rename('BurstRatings'))

users_with_bursts = (burst_stats > 10)  # >10 rapid ratings
print(f"    Users with >10 rapid ratings (<1hr gap): {users_with_bursts.sum():,}")
print(f"    → Rapid-fire rating bursts are a shilling attack signal")

fig, ax = plt.subplots(figsize=(7, 4))
burst_values = burst_stats.values
ax.hist(burst_values[burst_values < 100], bins=50, color='#E63946', edgecolor='white')
ax.axvline(10, color='black', linestyle='--', label='Burst threshold (10)')
ax.set_title("Figure 5.1 – Rating Burst Counts per User")
ax.set_xlabel("Number of Rapid Successive Ratings"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_1_rating_bursts.png")


  5.1 Rating Burst Detection (Temporal Clustering)
    Users with >10 rapid ratings (<1hr gap): 6,040
    → Rapid-fire rating bursts are a shilling attack signal
  [Saved] figures\fig5_1_rating_bursts.png


## Filler Item Detection

In [19]:
print("\n  5.2 Filler Item Ratings (Popular Item Coverage)")
popular_movies = item_pop[item_pop > item_pop.quantile(0.80)].index
filler_ratio = (ratings[ratings['MovieID'].isin(popular_movies)]
                .groupby('UserID').size() /
                ratings.groupby('UserID').size())
filler_ratio = filler_ratio.dropna()

high_filler = filler_ratio > 0.8
print(f"    Users with >80% of ratings on popular (top 20%) movies: {high_filler.sum():,}")
print(f"    → High filler ratio → shilling indicator (disguise via popular items)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(filler_ratio, bins=50, color='#6A0572', edgecolor='white', alpha=0.8)
ax.axvline(0.8, color='red', linestyle='--', label='>80% filler threshold')
ax.set_title("Figure 5.2 – Filler Item Ratio per User")
ax.set_xlabel("Fraction of Ratings on Popular (Top-20%) Movies"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_2_filler_ratio.png")


  5.2 Filler Item Ratings (Popular Item Coverage)
    Users with >80% of ratings on popular (top 20%) movies: 1,709
    → High filler ratio → shilling indicator (disguise via popular items)
  [Saved] figures\fig5_2_filler_ratio.png


## User Rating Deviation From Mean

In [20]:
print("\n  5.3 User Mean Rating Deviation from Global Mean")
global_mean = ratings['Rating'].mean()
user_mean_dev = np.abs(user_stats['AvgRating'] - global_mean)
high_dev = user_mean_dev > 1.5
print(f"    Global mean rating                    : {global_mean:.3f}")
print(f"    Users with |avg_rating - global| > 1.5: {high_dev.sum():,}")
print(f"    → Large deviations indicate biased/manipulative rating patterns")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(user_mean_dev, bins=50, color='#F77F00', edgecolor='white')
ax.axvline(1.5, color='red', linestyle='--', label='|Deviation| > 1.5')
ax.set_title("Figure 5.3 – Per-User Mean Rating Deviation from Global Mean")
ax.set_xlabel("Absolute Deviation"); ax.set_ylabel("Number of Users")
ax.legend()
save_fig("fig5_3_rating_deviation.png")


  5.3 User Mean Rating Deviation from Global Mean
    Global mean rating                    : 3.582
    Users with |avg_rating - global| > 1.5: 10
    → Large deviations indicate biased/manipulative rating patterns
  [Saved] figures\fig5_3_rating_deviation.png


## Composite Shalling Suspicion Score

In [21]:
print("\n  5.4 Composite Shilling Suspicion Score")
suspicion = pd.DataFrame(index=user_stats.index)
suspicion['HighFiller']   = (filler_ratio > 0.8).astype(int)
suspicion['LowVariance']  = (user_stats['StdRating'] < 0.3).astype(int)
suspicion['HighDeviation']= (user_mean_dev > 1.5).astype(int)
burst_flag = (burst_stats > 10).reindex(user_stats.index, fill_value=0).astype(int)
suspicion['BurstRating']  = burst_flag
suspicion['SuspicionScore'] = suspicion.sum(axis=1)

score_dist = suspicion['SuspicionScore'].value_counts().sort_index()
print(score_dist.to_string())
high_risk = (suspicion['SuspicionScore'] >= 3)
print(f"\n    High-risk users (score ≥ 3): {high_risk.sum():,} "
      f"({high_risk.mean()*100:.2f}% of users)")

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2DC653','#B0E0A8','#FCBA03','#FC6F03','#D00000'][:len(score_dist)]
ax.bar(score_dist.index, score_dist.values, color=colors, edgecolor='white', width=0.6)
ax.set_title("Figure 5.4 – Composite Shilling Suspicion Score Distribution")
ax.set_xlabel("Suspicion Score (0–4)"); ax.set_ylabel("Number of Users")
for i, (idx, val) in enumerate(score_dist.items()):
    ax.text(idx, val + 20, str(val), ha='center', fontsize=9)
save_fig("fig5_4_suspicion_score.png")


  5.4 Composite Shilling Suspicion Score
SuspicionScore
1    4319
2    1720
3       1

    High-risk users (score ≥ 3): 1 (0.02% of users)
  [Saved] figures\fig5_4_suspicion_score.png


# Sparsity Analysis

In [22]:
print("\n[6] SPARSITY ANALYSIS")
print("-" * 40)

n_users  = ratings['UserID'].nunique()
n_movies = ratings['MovieID'].nunique()
n_ratings = len(ratings)
density  = n_ratings / (n_users * n_movies) * 100
sparsity = 100 - density

print(f"    Users         : {n_users:,}")
print(f"    Movies        : {n_movies:,}")
print(f"    Ratings       : {n_ratings:,}")
print(f"    Matrix size   : {n_users:,} × {n_movies:,} = {n_users*n_movies:,}")
print(f"    Density       : {density:.4f}%")
print(f"    Sparsity      : {sparsity:.4f}%")
print(f"    → High sparsity amplifies both natural and malicious noise effects")


[6] SPARSITY ANALYSIS
----------------------------------------
    Users         : 6,040
    Movies        : 3,706
    Ratings       : 1,000,209
    Matrix size   : 6,040 × 3,706 = 22,384,240
    Density       : 4.4684%
    Sparsity      : 95.5316%
    → High sparsity amplifies both natural and malicious noise effects


# Cold-Start Analysis

In [23]:
cold_start_users  = (ratings.groupby('UserID').size() < 10).sum()
cold_start_movies = (ratings.groupby('MovieID').size() < 10).sum()
print(f"\n    Cold-start users  (< 10 ratings) : {cold_start_users:,}")
print(f"    Cold-start movies (< 10 ratings) : {cold_start_movies:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
user_counts = ratings.groupby('UserID').size()
movie_counts = ratings.groupby('MovieID').size()

axes[0].hist(user_counts, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(10, color='red', linestyle='--', label='Cold-start threshold (10)')
axes[0].set_title("Figure 6a – User Interaction Counts")
axes[0].set_xlabel("Ratings per User"); axes[0].set_ylabel("Count"); axes[0].legend()

axes[1].hist(movie_counts, bins=60, color='tomato', edgecolor='white')
axes[1].axvline(10, color='black', linestyle='--', label='Cold-start threshold (10)')
axes[1].set_title("Figure 6b – Movie Interaction Counts")
axes[1].set_xlabel("Ratings per Movie"); axes[1].set_ylabel("Count"); axes[1].legend()
plt.tight_layout()
save_fig("fig6_sparsity.png")


    Cold-start users  (< 10 ratings) : 0
    Cold-start movies (< 10 ratings) : 446
  [Saved] figures\fig6_sparsity.png


# Conceptual Analaysis - Noise Taxonomy

## Noise Indicator Correalation Matrix

In [24]:
print("\n  7.1 Noise Feature Correlation Matrix")
noise_features = pd.DataFrame({
    'AvgRating'      : user_stats['AvgRating'],
    'StdRating'      : user_stats['StdRating'],
    'NumRatings'     : user_stats['NumRatings'],
    'MeanDeviation'  : user_mean_dev,
    'FillerRatio'    : filler_ratio.reindex(user_stats.index),
    'SuspicionScore' : suspicion['SuspicionScore'],
}).dropna()

corr_matrix = noise_features.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title("Figure 7.1 – Noise Feature Correlation Heatmap")
plt.tight_layout()
save_fig("fig7_1_correlation_heatmap.png")


  7.1 Noise Feature Correlation Matrix
  [Saved] figures\fig7_1_correlation_heatmap.png


## High Suspicion vs Normal User Comparison

In [25]:
print("\n  7.2 High-Suspicion vs Normal Users – Feature Comparison")
noise_features['IsHighRisk'] = (suspicion['SuspicionScore'] >= 2).reindex(noise_features.index, fill_value=False)

comparison = noise_features.groupby('IsHighRisk')[
    ['AvgRating','StdRating','NumRatings','FillerRatio']].mean().round(3)
comparison.index = ['Normal Users', 'High-Risk Users']
print(comparison.to_string())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
features = ['AvgRating','StdRating','NumRatings','FillerRatio']
labels   = ['Avg Rating','Rating Std Dev','Num Ratings','Filler Ratio']
for i, (feat, lab) in enumerate(zip(features, labels)):
    vals = [noise_features[noise_features['IsHighRisk']==False][feat].mean(),
            noise_features[noise_features['IsHighRisk']==True ][feat].mean()]
    axes[i].bar(['Normal','High-Risk'], vals,
                color=['#2DC653','#D00000'], edgecolor='white', width=0.5)
    axes[i].set_title(f"Figure 7.2 – {lab}")
    axes[i].set_ylabel(lab)
plt.suptitle("Figure 7.2 – Feature Comparison: Normal vs High-Risk Users", y=1.02)
plt.tight_layout()
save_fig("fig7_2_user_comparison.png")


  7.2 High-Suspicion vs Normal Users – Feature Comparison
                 AvgRating  StdRating  NumRatings  FillerRatio
Normal Users         3.655      1.026     199.714        0.656
High-Risk Users      3.822      0.972      79.979        0.861
  [Saved] figures\fig7_2_user_comparison.png


# Final Noise Classification Summary

In [26]:
print("\n  7.3 Noise Classification Summary")
print("""
  ┌──────────────────────────────────────────────────────────────────────┐
  │             NOISE TAXONOMY IN MOVIELENS 1M DATASET                  │
  ├──────────────────────┬───────────────────────────────────────────────┤
  │  NATURAL NOISE       │  Indicators Found                            │
  │  (Benign)            │  • High rating variance per user             │
  │                      │  • Extreme raters (all-1s or all-5s)         │
  │                      │  • Low entropy users (narrow rating spread)  │
  │                      │  • Temporal rating drift (mood/context shift)│
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  MALICIOUS NOISE     │  Proxy Indicators Found                      │
  │  (Shilling Attacks)  │  • Rating bursts (rapid successive ratings)  │
  │                      │  • High filler item ratio (popular coverage) │
  │                      │  • Extreme deviation from global mean        │
  │                      │  • Composite high-suspicion score            │
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  DATA QUALITY        │  • Sparsity: ~95.53% missing entries         │
  │  CONCERNS            │  • Cold-start problem present                │
  │                      │  • Long-tail item popularity distribution    │
  └──────────────────────┴───────────────────────────────────────────────┘
""")


  7.3 Noise Classification Summary

  ┌──────────────────────────────────────────────────────────────────────┐
  │             NOISE TAXONOMY IN MOVIELENS 1M DATASET                  │
  ├──────────────────────┬───────────────────────────────────────────────┤
  │  NATURAL NOISE       │  Indicators Found                            │
  │  (Benign)            │  • High rating variance per user             │
  │                      │  • Extreme raters (all-1s or all-5s)         │
  │                      │  • Low entropy users (narrow rating spread)  │
  │                      │  • Temporal rating drift (mood/context shift)│
  ├──────────────────────┼───────────────────────────────────────────────┤
  │  MALICIOUS NOISE     │  Proxy Indicators Found                      │
  │  (Shilling Attacks)  │  • Rating bursts (rapid successive ratings)  │
  │                      │  • High filler item ratio (popular coverage) │
  │                      │  • Extreme deviation from global mean        

# Final Summary Table

In [27]:
print("\n[8] DATASET SUMMARY REPORT")
print("=" * 70)
summary = {
    "Total Ratings"                    : f"{n_ratings:,}",
    "Unique Users"                     : f"{n_users:,}",
    "Unique Movies"                    : f"{n_movies:,}",
    "Rating Scale"                     : "1–5 (integer)",
    "Matrix Density"                   : f"{density:.4f}%",
    "Matrix Sparsity"                  : f"{sparsity:.4f}%",
    "Mean Ratings/User"                : f"{user_activity.mean():.1f}",
    "Mean Ratings/Movie"               : f"{item_pop.mean():.1f}",
    "High-Variance Users (top 10%)"    : f"{high_variance_users.sum():,}",
    "Extreme Raters (all-high/all-low)": f"{all_high.sum() + all_low.sum():,}",
    "Burst Raters (>10 rapid)"         : f"{users_with_bursts.sum():,}",
    "High Filler Ratio Users (>80%)"   : f"{high_filler.sum():,}",
    "High-Risk Users (score≥3)"        : f"{high_risk.sum():,}",
    "Cold-Start Users (<10 ratings)"   : f"{cold_start_users:,}",
    "Cold-Start Movies (<10 ratings)"  : f"{cold_start_movies:,}",
}
for k, v in summary.items():
    print(f"  {k:<42}: {v}")

print("\n" + "=" * 70)
print("  All figures saved to ./figures/")
print("  Analysis complete ✓")
print("=" * 70)


[8] DATASET SUMMARY REPORT
  Total Ratings                             : 1,000,209
  Unique Users                              : 6,040
  Unique Movies                             : 3,706
  Rating Scale                              : 1–5 (integer)
  Matrix Density                            : 4.4684%
  Matrix Sparsity                           : 95.5316%
  Mean Ratings/User                         : 165.6
  Mean Ratings/Movie                        : 269.9
  High-Variance Users (top 10%)             : 604
  Extreme Raters (all-high/all-low)         : 13
  Burst Raters (>10 rapid)                  : 6,040
  High Filler Ratio Users (>80%)            : 1,709
  High-Risk Users (score≥3)                 : 1
  Cold-Start Users (<10 ratings)            : 0
  Cold-Start Movies (<10 ratings)           : 446

  All figures saved to ./figures/
  Analysis complete ✓


---
# Malicious Noise Injection
## Research Context
The MovieLens 1M dataset is a clean academic benchmark — it contains no confirmed malicious users.  
To evaluate the UNRR framework's ability to detect and mitigate shilling attacks, we inject **four standard attack types** into a 100K sample of the merged master dataset.

| Attack Type | Strategy | Goal |
|---|---|---|
| **Random Attack** | Random filler ratings + max score on target | Simple push attack |
| **Average Attack** | Item-mean filler ratings + max score on target | Hard-to-detect push |
| **Bandwagon Attack** | Popular item filler + max score on target | Exploit popularity bias |
| **Love/Hate Attack** | All 5s (push) or all 1s (nuke) | Extreme rating manipulation |

## Configuration & Setup

In [28]:
# ── Malicious Noise Injection — Configuration ────────────────────────────────
import random

INJECTION_CONFIG = {
    'sample_size'         : 100_000,   # Ratings to sample from master
    'random_seed'         : 42,        # Reproducibility
    'attack_percentage'   : 0.05,      # 5% of users will be fake attackers
    'filler_size'         : 50,        # Filler items rated per attacker
    'target_size'         : 5,         # Target items per attacker (push goal)
    'push_ratio'          : 0.5,       # 50% push / 50% nuke for love-hate
}

random.seed(INJECTION_CONFIG['random_seed'])
np.random.seed(INJECTION_CONFIG['random_seed'])

INJECT_OUTPUT_DIR = "injection_output"
os.makedirs(INJECT_OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  MALICIOUS NOISE INJECTION")
print("  Robust Recommender System — Group 17, CDU")
print("=" * 60)
print(f"\n  Configuration:")
for k, v in INJECTION_CONFIG.items():
    print(f"    {k:<25}: {v}")

  MALICIOUS NOISE INJECTION
  Robust Recommender System — Group 17, CDU

  Configuration:
    sample_size              : 100000
    random_seed              : 42
    attack_percentage        : 0.05
    filler_size              : 50
    target_size              : 5
    push_ratio               : 0.5


## Step 1 — Load Merged CSV & Sample 100K

In [29]:
# ── Load master_merged.csv and sample 100K ───────────────────────────────────
# We load the merged CSV saved earlier (master_merged.csv) as the base dataset.
# A 100K sample is taken for injection — manageable size for framework testing.

print("\n[INJECT-1] Loading master_merged.csv and sampling 100K...")

master_full = pd.read_csv("master_merged.csv")

# Sample 100K rows reproducibly
ml100k = master_full.sample(
    n=INJECTION_CONFIG['sample_size'],
    random_state=INJECTION_CONFIG['random_seed']
).reset_index(drop=True)

# Save clean 100K original as ground truth
original_path = os.path.join(INJECT_OUTPUT_DIR, "ml100k_original.csv")
ml100k.to_csv(original_path, index=False)

print(f"  master_merged.csv loaded  : {master_full.shape[0]:,} rows")
print(f"  100K sample created       : {len(ml100k):,} rows")
print(f"  Unique users              : {ml100k['UserID'].nunique():,}")
print(f"  Unique movies             : {ml100k['MovieID'].nunique():,}")
print(f"  Global mean rating        : {ml100k['Rating'].mean():.4f}")
print(f"  Clean original saved  → {original_path}")
print(f"\n  Columns available: {ml100k.columns.tolist()}")


[INJECT-1] Loading master_merged.csv and sampling 100K...
  master_merged.csv loaded  : 1,000,209 rows
  100K sample created       : 100,000 rows
  Unique users              : 5,970
  Unique movies             : 3,294
  Global mean rating        : 3.5826
  Clean original saved  → injection_output\ml100k_original.csv

  Columns available: ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'AgeGroup', 'OccupationName', 'Title', 'Genres']


## Step 2 — Prepare Attack Parameters

In [30]:
# ── Prepare attack parameters ────────────────────────────────────────────────
# Identify popular items (for bandwagon filler), target items (least-popular,
# simulate push attack on niche movies), and attacker count per type.

print("\n[INJECT-2] Preparing attack parameters...")

n_users_100k    = ml100k['UserID'].nunique()
global_mean_100k = ml100k['Rating'].mean()

# Item popularity in the 100K sample
item_pop_100k   = ml100k.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
all_items_100k  = ml100k['MovieID'].unique().tolist()
popular_items_100k = item_pop_100k.head(100).index.tolist()   # top 100 popular

# Target items — least popular movies (simulate push attack on niche items)
target_items_100k = item_pop_100k.tail(50).index.tolist()
target_items_100k = random.sample(
    target_items_100k,
    min(INJECTION_CONFIG['target_size'], len(target_items_100k))
)

# Attacker counts
n_attackers  = max(1, int(n_users_100k * INJECTION_CONFIG['attack_percentage']))
n_random     = max(1, int(n_attackers * 0.25))
n_average    = max(1, int(n_attackers * 0.25))
n_bandwagon  = max(1, int(n_attackers * 0.25))
n_lovehate   = n_attackers - n_random - n_average - n_bandwagon

# Fake UserIDs start above the maximum real UserID
fake_user_start = ml100k['UserID'].max() + 1

# Timestamp range for realistic fake timestamps
ts_series   = pd.to_datetime(ml100k['Timestamp']).astype(np.int64) // 10**9
ts_min_unix = int(ts_series.min())
ts_max_unix = int(ts_series.max())

print(f"  Real users in 100K        : {n_users_100k:,}")
print(f"  Total fake attackers      : {n_attackers} ({INJECTION_CONFIG['attack_percentage']*100:.0f}% of users)")
print(f"    Random Attack           : {n_random}")
print(f"    Average Attack          : {n_average}")
print(f"    Bandwagon Attack        : {n_bandwagon}")
print(f"    Love/Hate Attack        : {n_lovehate}")
print(f"  Target movies (push goal) : {target_items_100k}")
print(f"  Filler size per attacker  : {INJECTION_CONFIG['filler_size']}")
print(f"  Fake UserIDs start at     : {fake_user_start}")


[INJECT-2] Preparing attack parameters...
  Real users in 100K        : 5,970
  Total fake attackers      : 298 (5% of users)
    Random Attack           : 74
    Average Attack          : 74
    Bandwagon Attack        : 74
    Love/Hate Attack        : 76
  Target movies (push goal) : [397, 1504, 3116, 2977, 2063]
  Filler size per attacker  : 50
  Fake UserIDs start at     : 6041


## Step 3 — Attack Generation Functions

In [31]:
# ── Attack generation helper ─────────────────────────────────────────────────
def burst_timestamp():
    """All ratings within a 1-hour burst window — shilling temporal signature."""
    start = random.randint(ts_min_unix, ts_max_unix - 3600)
    return start

def make_profile(user_id, items, ratings_list, attack_type, is_filler_list, is_target_list, burst_ts):
    """Build a list of rating dicts for one fake user."""
    rows = []
    for item, rating, is_filler, is_target in zip(items, ratings_list, is_filler_list, is_target_list):
        rows.append({
            'UserID'      : user_id,
            'MovieID'     : item,
            'Rating'      : float(rating),
            'Timestamp'   : burst_ts + random.randint(0, 3600),
            'AttackType'  : attack_type,
            'IsFiller'    : is_filler,
            'IsTarget'    : is_target,
            'IsMalicious' : True
        })
    return rows


# ── ATTACK TYPE 1: Random Attack ──────────────────────────────────────────────
def random_attack(user_id):
    """
    Filler items get random ratings (1-5) to mimic normal user behaviour.
    Target items get maximum rating (5) to push them up the recommendation list.
    Easiest attack to detect due to random filler rating variance.
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    ts = burst_timestamp()
    items   = filler + target_items_100k
    ratings_vals = [float(random.randint(1,5)) for _ in filler] + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Random', fillers, targets, ts)


# ── ATTACK TYPE 2: Average Attack ─────────────────────────────────────────────
def average_attack(user_id):
    """
    Filler items rated near their item mean with small Gaussian noise.
    This makes the attacker look like a genuine user — harder to detect.
    Target items still receive maximum rating (5).
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    item_means = ml100k.groupby('MovieID')['Rating'].mean()
    ts = burst_timestamp()
    filler_ratings = []
    for item in filler:
        mean_r = item_means.get(item, global_mean_100k)
        r = round(float(np.clip(mean_r + np.random.normal(0, 0.3), 1, 5)) * 2) / 2
        filler_ratings.append(r)
    items   = filler + target_items_100k
    ratings_vals = filler_ratings + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Average', fillers, targets, ts)


# ── ATTACK TYPE 3: Bandwagon Attack ───────────────────────────────────────────
def bandwagon_attack(user_id):
    """
    Filler items are drawn exclusively from the top-100 popular movies.
    High ratings on popular items make the profile appear genuine and engaged.
    Most difficult to detect — exploits natural popularity bias in the system.
    """
    filler = random.sample([i for i in popular_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(popular_items_100k)))
    ts = burst_timestamp()
    filler_ratings = [float(random.choice([3,4,4,5,5])) for _ in filler]
    items   = filler + target_items_100k
    ratings_vals = filler_ratings + [5.0]*len(target_items_100k)
    fillers = [True]*len(filler) + [False]*len(target_items_100k)
    targets = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, 'Bandwagon', fillers, targets, ts)


# ── ATTACK TYPE 4: Love/Hate Attack ───────────────────────────────────────────
def lovehate_attack(user_id, is_push=True):
    """
    Push (Love): All ratings are 5 — artificially inflate target item scores.
    Nuke (Hate): All ratings are 1 — artificially suppress competitor items.
    Simplest attack — zero rating variance makes it the easiest to detect.
    """
    filler = random.sample([i for i in all_items_100k if i not in target_items_100k],
                           min(INJECTION_CONFIG['filler_size'], len(all_items_100k)-len(target_items_100k)))
    ts         = burst_timestamp()
    r_val      = 5.0 if is_push else 1.0
    label      = 'Love-Push' if is_push else 'Hate-Nuke'
    items      = filler + target_items_100k
    ratings_vals = [r_val]*len(items)
    fillers    = [True]*len(filler) + [False]*len(target_items_100k)
    targets    = [False]*len(filler) + [True]*len(target_items_100k)
    return make_profile(user_id, items, ratings_vals, label, fillers, targets, ts)

print("  All 4 attack functions defined ✓")
print("  Attack Type 1 : Random Attack")
print("  Attack Type 2 : Average Attack")
print("  Attack Type 3 : Bandwagon Attack")
print("  Attack Type 4 : Love/Hate Attack (Push + Nuke)")

  All 4 attack functions defined ✓
  Attack Type 1 : Random Attack
  Attack Type 2 : Average Attack
  Attack Type 3 : Bandwagon Attack
  Attack Type 4 : Love/Hate Attack (Push + Nuke)


## Step 4 — Generate & Inject All Fake Profiles

In [32]:
# ── Generate all fake profiles and inject into 100K dataset ─────────────────
print("\n[INJECT-3] Generating malicious profiles...")

all_fake = []
uid = fake_user_start

# Random attackers
for _ in range(n_random):
    all_fake.extend(random_attack(uid)); uid += 1

# Average attackers
for _ in range(n_average):
    all_fake.extend(average_attack(uid)); uid += 1

# Bandwagon attackers
for _ in range(n_bandwagon):
    all_fake.extend(bandwagon_attack(uid)); uid += 1

# Love/Hate attackers (split push/nuke)
n_push = int(n_lovehate * INJECTION_CONFIG['push_ratio'])
for i in range(n_lovehate):
    all_fake.extend(lovehate_attack(uid, is_push=(i < n_push))); uid += 1

fake_df = pd.DataFrame(all_fake)

print(f"  Fake attacker profiles    : {n_attackers}")
print(f"  Total fake ratings        : {len(fake_df):,}")
print(f"\n  Breakdown by attack type:")
print(fake_df.groupby('AttackType').agg(
    Profiles=('UserID','nunique'),
    Ratings=('Rating','count'),
    AvgRating=('Rating','mean')
).round(3).to_string())

# ── Label original 100K with noise columns ───────────────────────────────────
ml100k_labeled = ml100k.copy()
ml100k_labeled['AttackType']  = 'None'
ml100k_labeled['IsFiller']    = False
ml100k_labeled['IsTarget']    = False
ml100k_labeled['IsMalicious'] = False

# ── Combine real + fake into injected dataset ─────────────────────────────────
# Align columns — fake_df has subset of columns from master
fake_aligned = pd.DataFrame({
    'UserID'      : fake_df['UserID'],
    'MovieID'     : fake_df['MovieID'],
    'Rating'      : fake_df['Rating'],
    'Timestamp'   : pd.to_datetime(fake_df['Timestamp'], unit='s'),
    'Gender'      : 'U',           # Unknown — fake user
    'AgeGroup'    : 'Unknown',
    'OccupationName': 'Unknown',
    'Title'       : 'Unknown',
    'Genres'      : 'Unknown',
    'AttackType'  : fake_df['AttackType'],
    'IsFiller'    : fake_df['IsFiller'],
    'IsTarget'    : fake_df['IsTarget'],
    'IsMalicious' : fake_df['IsMalicious'],
})

injected_df = pd.concat([ml100k_labeled, fake_aligned], ignore_index=True)
injected_df = injected_df.sort_values(['UserID','Timestamp']).reset_index(drop=True)

# ── Save output files ─────────────────────────────────────────────────────────
injected_path = os.path.join(INJECT_OUTPUT_DIR, "ml100k_injected.csv")
fake_path     = os.path.join(INJECT_OUTPUT_DIR, "malicious_profiles.csv")

injected_df.to_csv(injected_path, index=False)
fake_df.to_csv(fake_path, index=False)

contamination = len(fake_df) / len(injected_df) * 100
print(f"\n[INJECT-4] Injection complete")
print(f"  Original 100K ratings     : {len(ml100k):,}")
print(f"  Injected fake ratings     : {len(fake_df):,}")
print(f"  Total injected dataset    : {len(injected_df):,}")
print(f"  Contamination rate        : {contamination:.2f}%")
print(f"\n  Saved → {injected_path}")
print(f"  Saved → {fake_path}")


[INJECT-3] Generating malicious profiles...
  Fake attacker profiles    : 298
  Total fake ratings        : 16,390

  Breakdown by attack type:
            Profiles  Ratings  AvgRating
AttackType                              
Average           74     4070      3.444
Bandwagon         74     4070      4.268
Hate-Nuke         38     2090      1.000
Love-Push         38     2090      5.000
Random            74     4070      3.242

[INJECT-4] Injection complete
  Original 100K ratings     : 100,000
  Injected fake ratings     : 16,390
  Total injected dataset    : 116,390
  Contamination rate        : 14.08%

  Saved → injection_output\ml100k_injected.csv
  Saved → injection_output\malicious_profiles.csv


## Step 5 — Visualise Attack Impact

In [33]:
# ── Figure: Rating distribution before vs after injection ────────────────────
print("\n[INJECT-5] Visualising attack impact...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Rating distribution comparison
rating_vals = [1.0, 2.0, 3.0, 4.0, 5.0]
orig_counts = [( ml100k['Rating'] == r).sum() for r in rating_vals]
inj_counts  = [(injected_df['Rating'] == r).sum() for r in rating_vals]

x = np.arange(len(rating_vals))
w = 0.35
axes[0].bar(x - w/2, orig_counts, w, label='Original', color='#4C72B0', edgecolor='white')
axes[0].bar(x + w/2, inj_counts,  w, label='Injected',  color='#D85A30', edgecolor='white')
axes[0].set_title("Fig I.1 – Rating Distribution: Before vs After Injection")
axes[0].set_xlabel("Rating Value"); axes[0].set_ylabel("Count")
axes[0].set_xticks(x); axes[0].set_xticklabels(['1','2','3','4','5'])
axes[0].legend()

# Plot 2: Attack type breakdown
atk_counts = fake_df.groupby('AttackType')['Rating'].count()
colors_atk = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']
axes[1].bar(atk_counts.index, atk_counts.values,
            color=colors_atk[:len(atk_counts)], edgecolor='white')
axes[1].set_title("Fig I.2 – Fake Ratings by Attack Type")
axes[1].set_xlabel("Attack Type"); axes[1].set_ylabel("Number of Fake Ratings")
axes[1].tick_params(axis='x', rotation=20)
for i, (idx, val) in enumerate(atk_counts.items()):
    axes[1].text(i, val + 10, str(val), ha='center', fontsize=9)

# Plot 3: Target item rating impact
target_before = [ml100k[ml100k['MovieID']==t]['Rating'].mean() for t in target_items_100k]
target_after  = [injected_df[injected_df['MovieID']==t]['Rating'].mean() for t in target_items_100k]
x_t = np.arange(len(target_items_100k))
axes[2].bar(x_t - w/2, target_before, w, label='Before', color='#4C72B0', edgecolor='white')
axes[2].bar(x_t + w/2, target_after,  w, label='After',  color='#D85A30', edgecolor='white')
axes[2].set_title("Fig I.3 – Target Item Avg Rating: Before vs After Attack")
axes[2].set_xlabel("Target Movie ID"); axes[2].set_ylabel("Average Rating")
axes[2].set_xticks(x_t)
axes[2].set_xticklabels([str(t) for t in target_items_100k], rotation=15)
axes[2].set_ylim(0, 5.5); axes[2].legend()

plt.tight_layout()
save_fig("fig_injection_impact.png")

# Print rating mean shift
print(f"  Original mean rating : {ml100k['Rating'].mean():.4f}")
print(f"  Injected mean rating : {injected_df['Rating'].mean():.4f}")
print(f"  Bias introduced      : {injected_df['Rating'].mean() - ml100k['Rating'].mean():+.4f}")


[INJECT-5] Visualising attack impact...
  [Saved] figures\fig_injection_impact.png
  Original mean rating : 3.5826
  Injected mean rating : 3.5689
  Bias introduced      : -0.0137


## Step 6 — Injection Summary Report

In [34]:
# ── Final injection summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  MALICIOUS NOISE INJECTION — SUMMARY REPORT")
print("=" * 60)

print(f"""
  DATASET
  -------
  Base dataset        : master_merged.csv (full 1M merge)
  Sample size         : {len(ml100k):,} ratings
  Real users          : {ml100k['UserID'].nunique():,}
  Real movies         : {ml100k['MovieID'].nunique():,}

  INJECTION
  ---------
  Total fake profiles : {n_attackers}
    Random Attack     : {n_random}
    Average Attack    : {n_average}
    Bandwagon Attack  : {n_bandwagon}
    Love/Hate Attack  : {n_lovehate}
  Total fake ratings  : {len(fake_df):,}
  Contamination rate  : {contamination:.2f}%
  Fake UserIDs        : {fake_user_start} – {uid-1}

  TARGET ITEMS (Push Attack Goal)
  --------------------------------""")

for t in target_items_100k:
    b = ml100k[ml100k['MovieID']==t]['Rating'].mean()
    a = injected_df[injected_df['MovieID']==t]['Rating'].mean()
    n = fake_df[fake_df['MovieID']==t].shape[0]
    print(f"  MovieID {t:<8}: {b:.3f} → {a:.3f}  (change: {a-b:+.3f}, {n} fake ratings)")

print(f"""
  OUTPUT FILES
  ------------
  injection_output/ml100k_original.csv    → Clean 100K ground truth
  injection_output/ml100k_injected.csv    → Contaminated dataset (use for UNRR testing)
  injection_output/malicious_profiles.csv → Fake profiles only

  KEY COLUMN: IsMalicious
  -----------------------
  False → genuine user rating
  True  → injected fake rating  (ground truth label for UNRR evaluation)

  AttackType column values:
  None / Random / Average / Bandwagon / Love-Push / Hate-Nuke
""")
print("=" * 60)
print("  Injection complete ✓  Ready for UNRR framework evaluation")
print("=" * 60)


  MALICIOUS NOISE INJECTION — SUMMARY REPORT

  DATASET
  -------
  Base dataset        : master_merged.csv (full 1M merge)
  Sample size         : 100,000 ratings
  Real users          : 5,970
  Real movies         : 3,294

  INJECTION
  ---------
  Total fake profiles : 298
    Random Attack     : 74
    Average Attack    : 74
    Bandwagon Attack  : 74
    Love/Hate Attack  : 76
  Total fake ratings  : 16,390
  Contamination rate  : 14.08%
  Fake UserIDs        : 6041 – 6338

  TARGET ITEMS (Push Attack Goal)
  --------------------------------
  MovieID 397     : 4.000 → 4.488  (change: +0.488, 298 fake ratings)
  MovieID 1504    : 4.000 → 4.488  (change: +0.488, 298 fake ratings)
  MovieID 3116    : 3.000 → 4.485  (change: +1.485, 298 fake ratings)
  MovieID 2977    : 5.000 → 4.492  (change: -0.508, 298 fake ratings)
  MovieID 2063    : 1.000 → 4.478  (change: +3.478, 298 fake ratings)

  OUTPUT FILES
  ------------
  injection_output/ml100k_original.csv    → Clean 100K ground tru